# PINN Lid-Driven Cavity Flow — Exploration

This notebook loads the trained model from `results/` and reproduces all
visualisations interactively.

**Run from the project root** (or adjust the `sys.path` cell below):
```
jupyter notebook notebooks/explore.ipynb
```

In [ ]:
import sys, os

# Add src/ to path — works whether launched from project root or notebooks/
for candidate in ['src', '../src']:
    p = os.path.join(os.getcwd(), candidate)
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
        break

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from model import MLP
from utils import (
    load_model, evaluate_on_grid,
    GHIA_Y, GHIA_U, GHIA_X, GHIA_V,
)

print('Imports OK')

## 1. Loss history

In [ ]:
loss = np.load('../results/loss_history.npy')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(loss)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Total loss')
axes[0].set_title('Training loss (linear scale x)')
axes[0].grid(True, alpha=0.4)

# Adam region only
adam_loss = loss[:20_000]
axes[1].semilogy(adam_loss)
axes[1].set_xlabel('Adam iteration')
axes[1].set_title('Adam phase only')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()
print(f'Adam final loss : {adam_loss[-1]:.4e}')
if len(loss) > 20_000:
    print(f'L-BFGS final loss: {loss[-1]:.4e}')

## 2. Load model and evaluate on grid

In [ ]:
model = load_model('../results/model.pt')
X, Y, U, V, P = evaluate_on_grid(model, n=100)
print(f'Grid shape : {X.shape}')
print(f'u range    : [{U.min():.3f}, {U.max():.3f}]')
print(f'v range    : [{V.min():.3f}, {V.max():.3f}]')
print(f'p range    : [{P.min():.3f}, {P.max():.3f}]')

## 3. Velocity and pressure fields

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

kw = dict(extent=[0, 1, 0, 1], origin='lower', aspect='equal')

im0 = axes[0].imshow(U.T, cmap='RdBu_r', **kw)
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('u-velocity')

im1 = axes[1].imshow(V.T, cmap='RdBu_r', **kw)
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('v-velocity')

im2 = axes[2].imshow(P.T, cmap='viridis', **kw)
plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Pressure')

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## 4. Streamlines

In [ ]:
x_lin = X[:, 0]
y_lin = Y[0, :]
speed = np.sqrt(U**2 + V**2)

fig, ax = plt.subplots(figsize=(6, 5))
strm = ax.streamplot(
    x_lin, y_lin, U.T, V.T,
    color=speed.T, cmap='viridis',
    density=1.5, linewidth=0.8, arrowsize=0.8,
)
plt.colorbar(strm.lines, ax=ax, label='Speed |u|')
ax.set_title('Streamlines  (Re = 100)')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.tight_layout()
plt.show()

## 5. Centreline profiles vs Ghia et al. (1982)

In [ ]:
n   = X.shape[0]
mid = n // 2

y_cl    = Y[mid, :]    # y values along x = x_lin[mid] ≈ 0.5
u_cl    = U[mid, :]    # u at x ≈ 0.5

x_cl    = X[:, mid]    # x values along y = y_lin[mid] ≈ 0.5
v_cl    = V[:, mid]    # v at y ≈ 0.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(u_cl, y_cl, 'b-', lw=2, label='PINN')
ax1.scatter(GHIA_U, GHIA_Y, color='red', s=50, zorder=5, label='Ghia et al. (1982)')
ax1.set_xlabel('u')
ax1.set_ylabel('y')
ax1.set_title('u-velocity at x = 0.5')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(x_cl, v_cl, 'b-', lw=2, label='PINN')
ax2.scatter(GHIA_X, GHIA_V, color='red', s=50, zorder=5, label='Ghia et al. (1982)')
ax2.set_xlabel('x')
ax2.set_ylabel('v')
ax2.set_title('v-velocity at y = 0.5')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Boundary-condition check

In [ ]:
import torch

def check_bc(model, n=200):
    t = torch.linspace(0, 1, n, dtype=torch.float64).reshape(-1, 1)
    walls = {
        'left   (u=v=0)' : torch.cat([torch.zeros_like(t), t], 1),
        'right  (u=v=0)' : torch.cat([torch.ones_like(t),  t], 1),
        'bottom (u=v=0)' : torch.cat([t, torch.zeros_like(t)], 1),
        'top    (u=1,v=0)': torch.cat([t, torch.ones_like(t)],  1),
    }
    with torch.no_grad():
        for name, xy in walls.items():
            out = model(xy).numpy()
            if 'top' in name:
                err_u = abs(out[:, 0] - 1).max()
            else:
                err_u = abs(out[:, 0]).max()
            err_v = abs(out[:, 1]).max()
            print(f'{name}: max|err_u| = {err_u:.3e}, max|err_v| = {err_v:.3e}')

check_bc(model)